# premiere league data

In [ ]:
import pandas as pd
epl_matches_loaded = pd.read_csv("data/epl_matches.csv")
display(epl_matches_loaded.head(40))
# list seasons and number of matches per season
season_counts = epl_matches_loaded['season'].value_counts().sort_index()
print("Number of matches per season:")
display(season_counts)


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Quick exploration of the Premier League matches dataset
print("Dataset shape:", epl_matches_loaded.shape)
print("\nColumns:")
print(epl_matches_loaded.columns.tolist())

print("\nMissing values per column:")
display(epl_matches_loaded.isna().sum().sort_values(ascending=False))

print("\nData types:")
display(epl_matches_loaded.dtypes)

if 'season' in epl_matches_loaded.columns:
    season_counts = epl_matches_loaded['season'].value_counts().sort_index()
    print("\nMatches per season:")
    display(season_counts)

if {'home_team', 'away_team', 'home_goals', 'away_goals'}.issubset(epl_matches_loaded.columns):
    explored = epl_matches_loaded.copy()
    explored['result'] = np.select(
        [explored['home_goals'] > explored['away_goals'], explored['home_goals'] < explored['away_goals']],
        ['Home win', 'Away win'],
        default='Draw'
    )
    explored['total_goals'] = explored['home_goals'] + explored['away_goals']

    print("\nResult distribution:")
    display(explored['result'].value_counts())

    print("\nMost frequent home teams:")
    display(explored['home_team'].value_counts().head(10))

    print("\nMost frequent away teams:")
    display(explored['away_team'].value_counts().head(10))

    print("\nHighest scoring matches:")
    display(explored.sort_values('total_goals', ascending=False)[['season', 'home_team', 'away_team', 'home_goals', 'away_goals', 'total_goals']].head(10))

    plt.figure(figsize=(10, 4))
    sns.countplot(data=explored, x='result', order=['Home win', 'Draw', 'Away win'])
    plt.title('Match Result Distribution')
    plt.xlabel('Result')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    sns.histplot(explored['total_goals'], bins=20, kde=True, color='steelblue')
    plt.title('Total Goals per Match')
    plt.xlabel('Total goals')
    plt.ylabel('Matches')
    plt.tight_layout()
    plt.show()

    team_stats = (
        explored.groupby('home_team')
        .agg(matches=('home_team', 'size'), avg_home_goals=('home_goals', 'mean'), avg_away_goals_allowed=('away_goals', 'mean'))
        .sort_values('matches', ascending=False)
        .head(10)
    )
    print("\nTop 10 home teams by match count:")
    display(team_stats)
else:
    numeric_cols = epl_matches_loaded.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        print("\nNumeric columns summary:")
        display(epl_matches_loaded[numeric_cols].describe().T)

    categorical_cols = epl_matches_loaded.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_cols:
        print("\nTop categories in the first few categorical columns:")
        for col in categorical_cols[:5]:
            print(f"\n{col}:")
            display(epl_matches_loaded[col].value_counts().head(10))
